**Create the Bronze notebook**

**Parameters
**
This cell is tagged as a **parameter cell** (Fabric-specific: toggle via
cell "..." menu → "Toggle parameter cell"). A calling Data Factory
pipeline can override these values at runtime without editing notebook
code — this is what makes the notebook safe to schedule.

In [10]:
# PARAMETERS CELL — tag this cell as a parameter cell in Fabric
# (cell "..." menu -> Toggle parameter cell)

processing_date = "2026-08-28"   # overridden by pipeline; defaults to today for manual runs
run_mode = "full"                 # "full" or "incremental" — controls Bronze source behavior
source_base_path = "Files/raw"    # base path for all raw file sources

StatementMeta(, b16e5be4-10aa-4ddc-890e-d4a52ee6ba51, 12, Finished, Available, Finished, False)

In [11]:
# ============================================================
# Bronze Layer Ingestion Notebook
# Reads all raw sources from Lakehouse Files and writes them
# as Bronze Delta tables with standardized audit columns.
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, BooleanType
from datetime import datetime

from datetime import datetime, timezone

BRONZE_INGESTION_TS = datetime.now(timezone.utc).isoformat()
PROCESSING_DATE = processing_date  # comes from the parameter cell above
print(f"Bronze ingestion run started at {BRONZE_INGESTION_TS} for processing_date={PROCESSING_DATE}, mode={run_mode}")
SOURCE_SYSTEM_FILES = "manual_upload_v1"  # per ADR-003


def add_audit_columns(df, source_name: str):
    """Standard audit columns applied to every Bronze table for lineage and debugging."""
    return (
        df.withColumn("_bronze_ingested_at", F.lit(BRONZE_INGESTION_TS).cast(TimestampType()))
          .withColumn("_source_system", F.lit(source_name))
          .withColumn("_source_file", F.input_file_name())
    )


print(f"Bronze ingestion run started at {BRONZE_INGESTION_TS}")

StatementMeta(, b16e5be4-10aa-4ddc-890e-d4a52ee6ba51, 13, Finished, Available, Finished, False)

Bronze ingestion run started at 2026-08-28T11:06:51.504721+00:00 for processing_date=2026-08-28, mode=full
Bronze ingestion run started at 2026-08-28T11:06:51.504721+00:00


**Postgres-sourced tables**

In [12]:
# ---- Customers ----
customers_raw = spark.read.option("header", True).csv(f"{source_base_path}/postgres/customers.csv")
customers_bronze = add_audit_columns(customers_raw, "postgres_customers")
customers_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_customers")
print(f"bronze_customers: {customers_bronze.count()} rows")

# ---- Products ----
products_raw = spark.read.option("header", True).csv(f"{source_base_path}/postgres/products.csv")
products_bronze = add_audit_columns(products_raw, "postgres_products")
products_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_products")
print(f"bronze_products: {products_bronze.count()} rows")

# ---- Orders ----
orders_raw = spark.read.option("header", True).csv(f"{source_base_path}/postgres/orders.csv")
orders_bronze = add_audit_columns(orders_raw, "postgres_orders")
orders_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_orders")
print(f"bronze_orders: {orders_bronze.count()} rows")

# ---- Order Items ----
order_items_raw = spark.read.option("header", True).csv(f"{source_base_path}/postgres/order_items.csv")
order_items_bronze = add_audit_columns(order_items_raw, "postgres_order_items")
order_items_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_order_items")
print(f"bronze_order_items: {order_items_bronze.count()} rows")

StatementMeta(, b16e5be4-10aa-4ddc-890e-d4a52ee6ba51, 14, Finished, Available, Finished, False)

bronze_customers: 520 rows
bronze_products: 100 rows
bronze_orders: 2003 rows
bronze_order_items: 6027 rows


— CSV loyalty source

In [13]:
loyalty_raw = spark.read.option("header", True).csv(f"{source_base_path}/loyalty/loyalty_export.csv")
loyalty_bronze = add_audit_columns(loyalty_raw, "csv_loyalty_export")
loyalty_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_loyalty")
print(f"bronze_loyalty: {loyalty_bronze.count()} rows")

StatementMeta(, b16e5be4-10aa-4ddc-890e-d4a52ee6ba51, 15, Finished, Available, Finished, False)

bronze_loyalty: 305 rows


**JSON reviews source (nested — read differently from flat CSVs)**

In [14]:
# multiLine=True is required because the JSON is a single pretty-printed
# object, not newline-delimited JSON (NDJSON)
reviews_raw = spark.read.option("multiLine", True).json(f"{source_base_path}/reviews/product_reviews.json")

# The actual reviews are nested inside a top-level "reviews" array — explode it
# into one row per review before flattening further in Silver (Phase 6).
reviews_exploded = reviews_raw.select(
    F.col("source").alias("_feed_source"),
    F.col("generated_at_utc"),
    F.explode("reviews").alias("review")
)

reviews_bronze = add_audit_columns(reviews_exploded, "json_reviews_feed")
reviews_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_reviews")
print(f"bronze_reviews: {reviews_bronze.count()} rows")
reviews_bronze.printSchema()

StatementMeta(, b16e5be4-10aa-4ddc-890e-d4a52ee6ba51, 16, Finished, Available, Finished, False)

bronze_reviews: 400 rows
root
 |-- _feed_source: string (nullable = true)
 |-- generated_at_utc: string (nullable = true)
 |-- review: struct (nullable = true)
 |    |-- helpful_votes: long (nullable = true)
 |    |-- product_id: long (nullable = true)
 |    |-- rating: long (nullable = true)
 |    |-- review_date: string (nullable = true)
 |    |-- review_id: string (nullable = true)
 |    |-- review_text: string (nullable = true)
 |    |-- reviewer: struct (nullable = true)
 |    |    |-- location: struct (nullable = true)
 |    |    |    |-- city: string (nullable = true)
 |    |    |    |-- country: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- verified_purchase: boolean (nullable = true)
 |    |-- tags: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |-- _bronze_ingested_at: timestamp (nullable = true)
 |-- _source_system: string (nullable = false)
 |-- _source_file: string (nullable = false)



**JSON exchange rates source**

In [15]:
rates_raw = spark.read.option("multiLine", True).json(f"{source_base_path}/exchange_rates/*.json")

rates_bronze = add_audit_columns(rates_raw, "rest_api_exchange_rates")
rates_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_exchange_rates")
print(f"bronze_exchange_rates: {rates_bronze.count()} rows")

StatementMeta(, b16e5be4-10aa-4ddc-890e-d4a52ee6ba51, 17, Finished, Available, Finished, False)

bronze_exchange_rates: 1 rows


**Validation summary**

In [16]:
tables = ["bronze_customers", "bronze_products", "bronze_orders", "bronze_order_items",
          "bronze_loyalty", "bronze_reviews", "bronze_exchange_rates"]

print("=" * 50)
print("BRONZE LAYER INGESTION SUMMARY")
print("=" * 50)
for t in tables:
    count = spark.table(t).count()
    print(f"{t:.<35}{count:>10} rows")

StatementMeta(, b16e5be4-10aa-4ddc-890e-d4a52ee6ba51, 18, Finished, Available, Finished, False)

BRONZE LAYER INGESTION SUMMARY
bronze_customers...................       520 rows
bronze_products....................       100 rows
bronze_orders......................      2003 rows
bronze_order_items.................      6027 rows
bronze_loyalty.....................       305 rows
bronze_reviews.....................       400 rows
bronze_exchange_rates..............         1 rows


**Schema Evolution: Demonstrating the Failure Mode**

Re-reading the loyalty source after an upstream change (a new
`referral_code` column was added). Writing this directly into the
existing `bronze_loyalty` table with default settings will fail, because
Delta enforces schema matching by default — this is intentional safety,
not a bug.

In [17]:
loyalty_raw_v2 = spark.read.option("header", True).csv(f"{source_base_path}/loyalty/loyalty_export.csv")
print("New schema:")
loyalty_raw_v2.printSchema()

try:
    loyalty_raw_v2.write.format("delta").mode("append").saveAsTable("bronze_loyalty")
    print("Write succeeded (unexpected).")
except Exception as e:
    print(f"Write FAILED as expected: {type(e).__name__}")
    print(str(e)[:300])

StatementMeta(, b16e5be4-10aa-4ddc-890e-d4a52ee6ba51, 19, Finished, Available, Finished, False)

New schema:
root
 |-- member_id: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- points_balance: string (nullable = true)
 |-- tier: string (nullable = true)
 |-- enrollment_date: string (nullable = true)
 |-- referral_code: string (nullable = true)

Write succeeded (unexpected).


**Schema Evolution: Explicit, Deliberate Merge**

Delta requires explicitly opting in to schema evolution via
`mergeSchema=true` — a deliberate choice, not a convenience default, so
adding a column is always a conscious decision reflected in the code.

In [18]:
loyalty_bronze_evolved = (
    loyalty_raw_v2.withColumn("_bronze_ingested_at", F.lit(BRONZE_INGESTION_TS).cast(TimestampType()))
                   .withColumn("_source_system", F.lit("csv_loyalty_export_v2"))
)

loyalty_bronze_evolved.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("bronze_loyalty")

print("Schema evolved successfully. New bronze_loyalty schema:")
spark.table("bronze_loyalty").printSchema()
print(f"\nRow count: {spark.table('bronze_loyalty').count()}")
spark.table("bronze_loyalty").select("member_id", "referral_code").show(5)

StatementMeta(, b16e5be4-10aa-4ddc-890e-d4a52ee6ba51, 20, Finished, Available, Finished, False)

Schema evolved successfully. New bronze_loyalty schema:
root
 |-- member_id: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- points_balance: string (nullable = true)
 |-- tier: string (nullable = true)
 |-- enrollment_date: string (nullable = true)
 |-- _bronze_ingested_at: timestamp (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- referral_code: string (nullable = true)


Row count: 305
+---------+-------------+
|member_id|referral_code|
+---------+-------------+
| LOY-1000|     REF-1000|
| LOY-1001|     REF-1001|
| LOY-1002|     REF-1002|
| LOY-1003|     REF-1003|
| LOY-1004|     REF-1004|
+---------+-------------+
only showing top 5 rows
